In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!nvidia-smi

In [ ]:
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
from huggingface_hub import login
login(token=hf_token)

In [ ]:
print(len(hf_token), repr(hf_token[:3]))

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")



In [ ]:
from huggingface_hub import login
login(token=hf_token)

In [ ]:
print(len(hf_token), repr(hf_token[:3]))

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


In [ ]:
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
print(len(hf_token), repr(hf_token[:3]))

In [ ]:
from huggingface_hub import login
login(token=hf_token.strip())

In [ ]:
!pip install -q wandb

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb
wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY").strip())

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("unsloth/llama-3-8b-bnb-4bit")
text = "Patient has chest pain and shortness of breath."
ids = tok.encode(text)

print(ids)
print(len(ids))
print(tok.convert_ids_to_tokens(ids))

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("unsloth/llama-3-8b-bnb-4bit")
text = "Patient presents with dyspnea and tachycardia."
ids = tok.encode(text)

print(ids)
print(len(ids))
print(tok.convert_ids_to_tokens(ids))

In [3]:
!python --version

Python 3.13.15


In [4]:
!pip install -q unsloth

In [5]:
import unsloth
print("unsloth imported OK")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth imported OK


In [6]:
import unsloth
from unsloth import FastLanguageModel
import torch

In [8]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.10.8: Fast Llama patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [9]:
used = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Using {used:.1f} GB of {total:.1f} GB")

Using 10.6 GB of 14.6 GB


In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.10.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).


In [11]:
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [12]:
used = torch.cuda.memory_allocated() / 1024**3
print(f"Using {used:.1f} GB")

Using 5.5 GB


In [13]:
import random, json
random.seed(42)

SYMPTOMS = ["chest pain", "shortness of breath", "fever", "cough", "headache",
            "nausea", "fatigue", "dizziness", "abdominal pain", "low back pain"]

DIAGNOSES = {
    "type 2 diabetes mellitus": "E11.9",
    "essential hypertension": "I10",
    "asthma": "J45.909",
    "community-acquired pneumonia": "J18.9",
    "urinary tract infection": "N39.0",
}

def make_example():
    age = random.randint(25, 85)
    sex = random.choice(["male", "female"])
    syms = random.sample(SYMPTOMS, k=random.randint(1, 3))
    dxs = random.sample(list(DIAGNOSES), k=random.randint(1, 2))
    days = random.randint(1, 7)
    templates = [
        f"{age} y/o {sex} c/o {', '.join(syms)}. PMH: {', '.join(dxs)}.",
        f"Pt is a {age}-year-old {sex} presenting with {' and '.join(syms)}. Known history of {' and '.join(dxs)}.",
        f"{sex.capitalize()}, {age}. Complains of {', '.join(syms)} x {days} days. Hx: {', '.join(dxs)}.",
    ]
    note = random.choice(templates)
    answer = {
        "symptoms": syms,
        "diagnoses": [{"name": d, "icd10": DIAGNOSES[d]} for d in dxs],
    }
    return note, answer

for _ in range(3):
    note, answer = make_example()
    print(note)
    print(json.dumps(answer, indent=2))
    print("-----")




Male, 65. Complains of headache x 2 days. Hx: essential hypertension.
{
  "symptoms": [
    "headache"
  ],
  "diagnoses": [
    {
      "name": "essential hypertension",
      "icd10": "I10"
    }
  ]
}
-----
Male, 31. Complains of fatigue, chest pain, abdominal pain x 2 days. Hx: essential hypertension.
{
  "symptoms": [
    "fatigue",
    "chest pain",
    "abdominal pain"
  ],
  "diagnoses": [
    {
      "name": "essential hypertension",
      "icd10": "I10"
    }
  ]
}
-----
Pt is a 63-year-old male presenting with cough and abdominal pain and fatigue. Known history of community-acquired pneumonia.
{
  "symptoms": [
    "cough",
    "abdominal pain",
    "fatigue"
  ],
  "diagnoses": [
    {
      "name": "community-acquired pneumonia",
      "icd10": "J18.9"
    }
  ]
}
-----


In [14]:
SYSTEM_PROMPT = (
    "You are a clinical information extraction assistant. "
    "Read the clinical note and return ONLY valid JSON with two keys: "
    '"symptoms" (a list of strings) and "diagnoses" '
    '(a list of objects, each with "name" and "icd10").'
)

def to_chatml(note, answer):
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": note},
        {"role": "assistant", "content": json.dumps(answer)},
    ]}

def is_valid(example):
    try:
        reply = example["messages"][2]["content"]
        parsed = json.loads(reply)
        return "symptoms" in parsed and "diagnoses" in parsed
    except Exception:
        return False

raw = [to_chatml(*make_example()) for _ in range(1000)]
good = [ex for ex in raw if is_valid(ex)]
print(len(raw), len(good))
print(json.dumps(good[0], indent=2))

1000 1000
{
  "messages": [
    {
      "role": "system",
      "content": "You are a clinical information extraction assistant. Read the clinical note and return ONLY valid JSON with two keys: \"symptoms\" (a list of strings) and \"diagnoses\" (a list of objects, each with \"name\" and \"icd10\")."
    },
    {
      "role": "user",
      "content": "Pt is a 76-year-old male presenting with fatigue. Known history of asthma and essential hypertension."
    },
    {
      "role": "assistant",
      "content": "{\"symptoms\": [\"fatigue\"], \"diagnoses\": [{\"name\": \"asthma\", \"icd10\": \"J45.909\"}, {\"name\": \"essential hypertension\", \"icd10\": \"I10\"}]}"
    }
  ]
}


In [15]:
from datasets import Dataset

seen = set()
unique = []
for ex in good:
    note = ex["messages"][1]["content"]
    if note not in seen:
        seen.add(note)
        unique.append(ex)
print(len(good), len(unique))

ds = Dataset.from_list(unique)
split1 = ds.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)
train_ds, val_ds, test_ds = split1["train"], split2["train"], split2["test"]
print(len(train_ds), len(val_ds), len(test_ds))

1000 1000
800 100 100


In [16]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

def to_text(batch):
    texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]
    return {"text": texts}

train_ds = train_ds.map(to_text, batched=True)
val_ds = val_ds.map(to_text, batched=True)
test_ds = test_ds.map(to_text, batched=True)

print(train_ds[0]["text"])

[unsloth.chat_templates|WARNING]Unsloth: Will map <|im_end|> to EOS = <|end_of_text|>.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_m7ln9o7q/tokenizer_config.json.


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

<|im_start|>system
You are a clinical information extraction assistant. Read the clinical note and return ONLY valid JSON with two keys: "symptoms" (a list of strings) and "diagnoses" (a list of objects, each with "name" and "icd10").<|im_end|>
<|im_start|>user
56 y/o female c/o nausea, chest pain. PMH: essential hypertension, type 2 diabetes mellitus.<|im_end|>
<|im_start|>assistant
{"symptoms": ["nausea", "chest pain"], "diagnoses": [{"name": "essential hypertension", "icd10": "I10"}, {"name": "type 2 diabetes mellitus", "icd10": "E11.9"}]}<|im_end|>



In [17]:
lengths = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"]]
print("longest:", max(lengths))
print("average:", round(sum(lengths) / len(lengths)))
print("over 2048:", sum(l > 2048 for l in lengths))

longest: 189
average: 158
over 2048: 0


In [18]:
import os
os.makedirs("data", exist_ok=True)
train_ds.to_json("data/train.jsonl")
val_ds.to_json("data/val.jsonl")
test_ds.to_json("data/test.jsonl")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

117772